# 03 - Feature engineering and selection

**Goal:** turn the findings of notebooks 01 and 02 into features, and decide which columns to keep.

- Uses the clean data saved by `01_data_cleaning.ipynb` (run it first).
- Same rules as 01: `spend_next_90d` is never a feature, and anything learned from the data is fitted on train only.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from scipy.stats import spearmanr
import sys
import importlib
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

import src.data as data_module
importlib.reload(data_module)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_DIR = "../data"
TARGET, SPEND = "repeat_purchase_90d", "spend_next_90d"
BLUE, ORANGE, GREY, RED = "#2f6690", "#d9822b", "#9aa5b1", "#c0392b"

# Same plot style as notebook 01
pio.templates["eda"] = go.layout.Template(layout=dict(
    font=dict(size=13), title=dict(font=dict(size=17)), height=450,
    margin=dict(t=80, l=60, r=30, b=50), hoverlabel=dict(bgcolor="white")))
pio.templates.default = "plotly_white+eda"

# Cleaned data saved at the end of notebook 01 (run 01 first)
# The categorical labels are numbers stored as text ("7"): read them as text, not as numbers
CAT_COLS = ["payment_type", "salesperson", "commercial_zone", "transport_method", "warehouse", "document_series"]
df = pd.read_csv(f"{DATA_DIR}/processed/train_clean.csv", parse_dates=["purchase_date"], dtype={c: str for c in CAT_COLS})
test = pd.read_csv(f"{DATA_DIR}/processed/test_clean.csv", parse_dates=["purchase_date"], dtype={c: str for c in CAT_COLS})
print("train:", df.shape, "| test:", test.shape)

train: (6469, 40) | test: (986, 38)


## 1. Feature ideas from notebooks 01 and 02

| Feature | Idea | Where it came from |
|---|---|---|
| `is_first_purchase` | 1 when `prior_purchase_count` = 0. On a 1st purchase the history columns are empty *by definition* (no history, not unknown): fill them with a constant, and the flag tells the model what that constant means. 1st purchases repeat 33%, returning customers 58%. | 02: Sections 1 and 4 |
| `gap_ratio` = `days_since_previous_purchase / prior_mean_gap_days` | Is the customer late compared with their usual rhythm? Tested in Section 2: weak on its own. | 02: Section 4 (recency and usual gap) |
| `month` | Month of the purchase (one-hot, or sine/cosine). March and November repeat about 45%; January, May and September about 70%. | 02: Section 3 |
| `country`, `island_missing` | From `clientes.csv`. CT5 repeats 39%, CT10 57%; `island` filled 64% vs 52%. Group small countries into "other". | 02: Section 7 |
| `vat_rate`, `margin`, `cost_is_zero`, `discount_share` | Ratios from the invoice columns (Section 3 below). | 01: Section 4 |
| `log1p` of the money columns | Values are very skewed; helps linear models. | 02: Sections 2 and 5 |

**To drop:** `random_noise` (already dropped in 01), `document_series`, the address columns of `clientes.csv` and `customer_type` (notebook 02, Sections 6 and 7).

## 2. Is the customer late compared with their usual rhythm? (`gap_ratio`)

**Idea:** "60 days since the last purchase" means different things for different clients. For someone who buys every 10 days it is very late; for someone who buys every 90 days it is normal. So we compare each client with **their own habit**:

`gap_ratio` = `days_since_previous_purchase` / `prior_mean_gap_days`

| client | usually buys every | last purchase | gap_ratio | meaning |
|---|---|---|---|---|
| A | 30 days | 15 days ago | 0.5 | early |
| B | 30 days | 30 days ago | 1 | on time |
| C | 30 days | 120 days ago | 4 | very late, maybe loosing the client |

**Expected:** the later the client (higher ratio), the less likely they come back.

Notes: the ratio is empty on 1st and 2nd purchases (no average gap yet). We divide by `prior_mean_gap_days + 1` so a gap of 0 days doesn't break the division.

The test has three parts:
1. Repeat rate per ratio band.
2. Clients with the **same** days since their last purchase, but a different ratio. Does being late still matter?
3. Correlation with the target: the ratio vs the two columns it is made from.

In [3]:
mean_gap = df["prior_mean_gap_days"]
valid_mean_gap = df["prior_purchase_count"].ge(2) & mean_gap.gt(0)

gap_ratio = df["days_since_previous_purchase"].div(
    mean_gap.where(valid_mean_gap)
)
has_ratio = gap_ratio.notna()

print(f"Rows with a defined gap ratio: {has_ratio.sum()} of {len(df)}")
# 1. Repeat rate per ratio band
ratio_band = pd.cut(gap_ratio, [0, 0.5, 1, 2, 4, np.inf], right=False,
                    labels=["< 0.5 (early)", "0.5-1", "1-2 (late)", "2-4", "4+ (very late)"])
print("\n1. Repeat rate per gap_ratio band")
display(df.groupby(ratio_band, observed=True)[TARGET].agg(["mean", "size"]).round(3)
          .rename(columns={"mean": "repeat rate", "size": "rows"}).rename_axis("gap_ratio"))

# 2. Same days since the last purchase (rows), on time vs late (columns)
days_band = pd.cut(df["days_since_previous_purchase"], [0, 30, 90, 180, np.inf], right=False,
                   labels=["0-29 days", "30-89 days", "90-179 days", "180+ days"])
late = pd.cut(gap_ratio, [0, 1, 2, np.inf], right=False, labels=["on time (< 1)", "late (1-2)", "very late (2+)"])
print("\n2. Repeat rate: same days since the last purchase, different gap_ratio")
display(df[has_ratio].groupby([days_band, late], observed=True)[TARGET].mean().unstack().round(2)
          .rename_axis(index="days since last purchase", columns="gap_ratio"))
print("Typical usual gap (median prior_mean_gap_days) per group:")
display(df[has_ratio].groupby([days_band, late], observed=True)["prior_mean_gap_days"].median().unstack().round(0)
          .rename_axis(index="days since last purchase", columns="gap_ratio"))

# 3. Correlation with the target, on the same rows
print("\n3. Spearman correlation with the target (same rows):")
for name, values in [("days_since_previous_purchase", df["days_since_previous_purchase"]),
                     ("prior_mean_gap_days", df["prior_mean_gap_days"]), ("gap_ratio", gap_ratio)]:
    print(f"  {name:30s} {spearmanr(values[has_ratio], df.loc[has_ratio, TARGET], nan_policy='omit')[0]:.2f}")

Rows with a defined gap ratio: 5421 of 6469

1. Repeat rate per gap_ratio band


,repeat rate,rows
gap_ratio,,
< 0.5 (early),0.623,1835
0.5-1,0.642,1322
1-2 (late),0.565,1397
2-4,0.581,644
4+ (very late),0.439,223



2. Repeat rate: same days since the last purchase, different gap_ratio


gap_ratio,on time (< 1),late (1-2),very late (2+)
days since last purchase,,,
0-29 days,0.72,0.95,0.89
30-89 days,0.63,0.78,0.79
90-179 days,0.32,0.42,0.61
180+ days,0.22,0.39,0.38


Typical usual gap (median prior_mean_gap_days) per group:


gap_ratio,on time (< 1),late (1-2),very late (2+)
days since last purchase,,,
0-29 days,52.0,11.0,7.0
30-89 days,85.0,48.0,27.0
90-179 days,146.0,86.0,46.0
180+ days,365.0,165.0,87.0



3. Spearman correlation with the target (same rows):
  days_since_previous_purchase   -0.32
  prior_mean_gap_days            -0.42
  gap_ratio                      -0.05


### Key insight - Section 2

**Found**

- `gap_ratio` was defined for **5,421 of 6,469 purchases**.
- Overall, purchases in the **4+ ratio band** had a repeat rate of **43.9%**, compared with **62.3%–64.2%** in the two bands below 1.
- Within the **90–179 day recency band**, the repeat rate was **32%** for ratios below 1 and **61%** for ratios of 2 or more. The median historical average gap was **146 days** in the first group and **46 days** in the second. This suggests that customers in the higher-ratio group typically bought more often.
- On the same rows, `gap_ratio` had a weaker Spearman correlation with repeat purchase (**−0.05**) than `prior_mean_gap_days` (**−0.42**) and `days_since_previous_purchase` (**−0.32**).

**For feature engineering**

- Use `days_since_previous_purchase` and `prior_mean_gap_days` as separate baseline features.
- Test `gap_ratio` as an additional feature on the same validation splits. Keep it if it improves validation performance.

## 3. Ratios from the invoice columns

- **Margin based on recorded cost** is `(net value - recorded cost) / net value`.
- **Effective tax rate** is `recorded tax / net value`.

Both ratios require a positive net value. The margin calculation also requires a positive recorded cost. Purchases with zero recorded cost form a separate group in the margin table.

We use these temporary calculations to compare repeat rates across bands and report the number of purchases in each band. The repeat rate is the share of purchases followed by another purchase within 90 days. The overall training repeat rate is **56.0%**.


In [4]:
net_value = df["Total Liquido"]
net = net_value.where(net_value.gt(0))
cost = df["Custo"]

cost_is_zero = cost.eq(0)
margin = net.sub(cost).div(net).where(cost.gt(0))
tax_rate = df["Total Imp_"].div(net)

print(
    f"Zero recorded cost: {cost_is_zero.sum()} | "
    f"Losses: {margin.lt(0).sum()} | "
    f"Break-even: {margin.eq(0).sum()}"
)

# Four common rounded effective tax rates in training.
rounded_tax = tax_rate.round(2)
vat_band = (
    rounded_tax.map({0.0: "0%", 0.13: "13%", 0.22: "22%", 0.23: "23%"})
    .fillna("Other")
    .mask(rounded_tax.isna(), "Missing")
)

margin_band = pd.cut(
    margin,
    bins=[-np.inf, 0, 0.2, 0.3, 0.4, np.inf],
    labels=["Loss", "(0%, 20%]", "(20%, 30%]", "(30%, 40%]", "Above 40%"],
).cat.add_categories(["Break-even", "Zero recorded cost", "Missing"])

margin_band = margin_band.mask(margin.eq(0), "Break-even")
margin_band = margin_band.mask(cost_is_zero, "Zero recorded cost")
margin_band = margin_band.fillna("Missing")

for name, band in [
    ("Effective tax band", vat_band),
    ("Margin band", margin_band),
]:
    display(
        df.groupby(band, observed=True)[TARGET]
        .agg(["mean", "size"])
        .rename(columns={"mean": "repeat rate", "size": "rows"})
        .round(3)
        .rename_axis(name)
    )

print(f"Overall repeat rate: {df[TARGET].mean():.1%}")

Zero recorded cost: 486 | Losses: 39 | Break-even: 2


,repeat rate,rows
Effective tax band,,
0%,0.390,213
13%,0.701,221
22%,0.678,115
23%,0.565,5641
Other,0.416,279


,repeat rate,rows
Margin band,,
Loss,0.744,39
"(0%, 20%]",0.640,114
"(20%, 30%]",0.606,381
"(30%, 40%]",0.427,1362
Above 40%,0.582,4085
Break-even,1.000,2
Zero recorded cost,0.677,486


Overall repeat rate: 56.0%


### Key insight - Section 3

**Found**

- Repeat rates varied across **effective tax bands**. The rate was **39.0%** in the 0% band, **70.1%** in the 13% band, and **67.8%** in the 22% band. The overall rate was **56.0%**.
- Country CT5 accounted for **193 of the 213 purchases** in the 0% tax band. The other **20 purchases** came from other countries.
- Repeat rates varied across **margin bands**. The 30–40% band had a repeat rate of **42.7%**, compared with **58.2%** above 40%. The **39 loss cases** had a repeat rate of **74.4%**, and there were **2 break-even purchases**.
- **486 purchases (7.5%)** had zero recorded cost. Their repeat rate was **67.7%**.

**For feature engineering**

- After inspecting the original predictors, test `tax_rate`, `margin`, and `discount_share` as candidate features. `discount_share` is total discounts divided by gross invoice value.
- Add `cost_is_zero` to identify purchases with zero recorded cost and leave their `margin` as `NaN`. Fit any imputation within each training fold.
- Compare effective tax rate and country separately and together during validation to see what each contributes.

## 4. Which original numeric predictors are associated with `repeat_purchase_90d`?

We use Spearman correlation to measure monotonic associations with repeat purchase. A positive correlation means purchases followed by a repeat tend to have higher feature values.

Each correlation uses rows where both the predictor and target are available, so sample sizes can differ across predictors. A feature with a non-monotonic relationship can have a weak correlation and still improve model performance. We use validation to assess its contribution.

In [5]:
num_cols = df.select_dtypes("number").columns.drop(["ID", "customer_id", TARGET, SPEND], errors="ignore")
corr_target = pd.Series({c: spearmanr(df[c], df[TARGET], nan_policy="omit")[0] for c in num_cols}).sort_values()

colors = [BLUE if v > 0 else ORANGE for v in corr_target]
fig = go.Figure(go.Bar(x=corr_target.values, y=corr_target.index, orientation="h", marker_color=colors,
                       hovertemplate="%{y}: %{x:.3f}<extra></extra>"))
fig.add_vline(x=0, line_color="black", line_width=1)
fig.update_layout(title="Spearman correlation with repeat_purchase_90d", xaxis_title="correlation", height=800)
fig.show()

### Key insight - Section 4

**Found**
- The signal is in the **customer's history**: purchases in recent windows, recency and the average gap (correlation 0.35-0.44).
- The current purchase barely matters (below 0.1), except line discounts (0.2).

**For feature selection**
- Keep the history columns: they carry most of the signal.
- The current-purchase columns are candidates to drop. Confirm with a model-based method (e.g. permutation importance), since Spearman misses curved effects.

## 5. Which original numeric predictors are strongly correlated?

We use Spearman correlation to identify predictors with similar or opposite rankings. Pairs with an absolute correlation above 0.90 may contain overlapping information.

For these pairs, we compare validation performance using both predictors and using each one separately. This helps us decide whether a smaller feature set preserves performance.

In [6]:
corr_matrix = df[num_cols].corr(method="spearman")
fig = px.imshow(corr_matrix, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="equal")
fig.update_layout(title="Spearman correlation between numeric predictors", height=900, width=1000)
fig.show()

pairs = corr_matrix.where(np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)).stack()
print("Pairs with |rho| > 0.9:")
print(pairs[pairs.abs() > 0.9].sort_values(ascending=False).round(3).to_string())

Pairs with |rho| > 0.9:
Total a Pagar             Total Liquido               1.000
Total Bruto               Total Liquido               0.991
Total a Pagar             Total Bruto                 0.991
purchase_count_30d        spend_30d                   0.965
prior_std_purchase_value  prior_max_purchase_value    0.936
purchase_count_90d        spend_90d                   0.909
prior_purchase_count      prior_total_spend           0.903
spend_180d                spend_365d                  0.900


### Key insight - Section 5

**Found** - several pairs are near copies (correlation > 0.9):
- the money columns of the invoice (`Total a Pagar`, `Total Liquido`, `Total Bruto`),
- count vs spend in the same window (30 and 90 days),
- `spend_180d` vs `spend_365d`,
- `prior_purchase_count` vs `prior_total_spend`,
- `prior_std_purchase_value` vs `prior_max_purchase_value`.

**For feature selection**
- Keep one money column (e.g. `Total Liquido`) and drop the others.
- From each other pair, keep the one more related to the target (Section 4), or let the selection method decide.

## 6. Which columns are related to `spend_next_90d`?

Only customers who came back (target = 1) have a spend value, so the correlation uses those rows. It is shown next to the correlation with repeat from Section 4.

In [7]:
returned = df[(df[TARGET] == 1) & df[SPEND].notna()]
corr_spend = pd.Series({c: spearmanr(returned[c], returned[SPEND], nan_policy="omit")[0] for c in num_cols})
both = pd.DataFrame({"repeat (classification)": corr_target, "spend (regression, returners)": corr_spend})
both = both.reindex(both["spend (regression, returners)"].abs().sort_values().index)

fig = go.Figure()
for name, color in [("repeat (classification)", GREY), ("spend (regression, returners)", BLUE)]:
    fig.add_bar(x=both[name], y=both.index, orientation="h", name=name, marker_color=color)
fig.add_vline(x=0, line_color="black", line_width=1)
fig.update_layout(title="Same predictors, two targets (Spearman)", barmode="group", height=900)
fig.show()

### Key insight - Section 6

**Found**
- Spend depends mostly on **past spend**: `spend_365d` (0.60), `spend_180d` (0.56), `prior_total_spend` (0.52).
- The current purchase value matters a bit more than for repeat (0.16 vs 0.05), but it is still secondary.
- The features rank in a similar order for both targets.

**For feature selection**
- Use the same feature set for both models: simpler, and the two predictions stay consistent.

## Summary

**Create**
- `is_first_purchase` (+ fill the "no history" NaN with a constant)
- `gap_ratio` only if it improves validation (Section 2: weak on its own)
- `month`, `country` (grouped), `island_missing`
- `vat_rate` (or `country`, not both), `margin`, `cost_is_zero`, `discount_share`

**Drop**
- `random_noise` (done in 01), `document_series`, the address columns of `clientes.csv`, `customer_type`
- the redundant money columns (keep one, e.g. `Total Liquido`)
- one column of each near-copy pair, unless the selection method shows both help

**Next:** build these features inside the sklearn `Pipeline` (fitted on train only) and confirm the selection with time-based validation.

## Summary

**Create candidate features**

- Use `add_candidate_features()` to create `gap_ratio`, `margin`, `tax_rate`, `cost_is_zero`, and `discount_share` for train and test.
- Test `is_first_purchase` and `month` as additional features. Test grouped country and `island_missing` if we join `clientes.csv`.

**Decide which features to keep**

- Compare models with and without `random_noise` and `document_series`.
- Compare `tax_rate` and country individually and together.
- Test smaller sets of correlated predictors, including the monetary columns.
- Exclude `customer_type` if using the customer table, since it is entirely missing. Leave raw addresses out of the initial feature set.

**Validate**

Use the same chronological splits and model settings for each comparison. Fit imputation, encoding, and rare-country grouping within each training fold. Choose features separately for repeat classification and spending regression.

In [12]:
from src.data import add_candidate_features


train_engineered = add_candidate_features(df)
test_engineered = add_candidate_features(test)

new_cols = [
    "gap_ratio",
    "margin",
    "tax_rate",
    "cost_is_zero",
    "discount_share",
]

display(train_engineered[new_cols].head())

,gap_ratio,margin,tax_rate,cost_is_zero,discount_share
0,1.680851,0.662590,0.229994,0,0.111861
1,2.623656,0.405679,0.230004,0,0.041264
2,1.332779,0.274109,0.229762,0,0.300000
3,NaN,0.414066,0.230000,0,0.000000
4,1.369128,0.630564,0.230002,0,0.117507


In [ ]:
# Run after creating train_engineered and test_engineered.
columns_to_drop = [
    "random_noise",
    "document_series",
    "customer_type",

    # Keep Total Liquido as the invoice total.
    "Total a Pagar",
"Total Liquido",

    # Provisional choices for the correlated pairs.
    "spend_30d",                 # Keep purchase_count_30d
    "spend_90d",                 # Keep purchase_count_90d
    "spend_180d",                # Keep spend_365d
    "prior_total_spend",         # Keep prior_purchase_count
    "prior_std_purchase_value", # Keep prior_max_purchase_value
]

train_selected = train_engineered.drop(columns=columns_to_drop, errors="ignore")
test_selected = test_engineered.drop(columns=columns_to_drop, errors="ignore")